# Retail Sales Analysis with SQL and SQLite

## Project overview

This project uses SQL to analyse a reproducible synthetic retail database, investigating monthly revenue, product and category performance, customer purchasing, geographic differences, customer concentration, and stocked products without sales.

**Tools:** SQLite, Python, pandas, and Jupyter Notebook. Python generates the data and displays results; SQL performs the analysis.

**Analysis period:** 1 January–31 December 2026. The source database contains simulated orders from 2025 and 2026. All monetary amounts are expressed in South African rand (R).

## Dataset and relationships

| Table | Rows | Purpose |
|---|---:|---|
| Customers | 100 | Customer identifiers, names, and cities |
| Products | 12 | Product categories, current prices, and stock snapshot |
| Orders | 500 | Customer orders, dates, and order totals |
| Order_Items | 1,237 | Products, quantities, and selling prices on each order |

Customers link to Orders through `Customer_ID`. Orders link to Order_Items through `Order_ID`, and Products link to Order_Items through `Product_ID`.

## Business questions

- How do revenue, order volume, and average order value vary by month?
- Which products and categories lead revenue and unit sales?
- How do customer participation and spending differ by city?
- How concentrated is city revenue among the highest-spending customers?
- Which stocked products have no sales during the analysis period?

## How to run

Run the notebook from top to bottom in a Python environment with pandas and SQLite window-function support. The setup creates `retail_sql_project.db` in the working directory. The generator uses seed 42 and inserts data only when all four tables are empty; existing data is retained. Use a fresh project folder to reproduce the original generated dataset.

## Metric definitions

- **Revenue:** order totals for order/customer analysis; quantity × order-item selling price for product/category analysis.
- **Average order value:** revenue divided by order count.
- **Customer order rate:** customers with at least one 2026 order divided by all customers in the group.
- **Revenue share:** revenue divided by the relevant overall or city total, calculated before filtering ranks.
- **Unsold stock value:** stock quantity × current selling price; this is not inventory cost or guaranteed future revenue.

The dataset is synthetic, including simulated future dates relative to its creation. Results describe the generated scenario, not real market demand.


## 1. Connect to the database

In [1]:
import sqlite3
import pandas as pd
import random
from datetime import date, timedelta

conn = sqlite3.connect("retail_sql_project.db")
conn.execute("PRAGMA foreign_keys = ON;")

print("Connected to retail_sql_project.db")

Connected to retail_sql_project.db


## 2. Create tables and constraints

In [2]:
conn.executescript("""
CREATE TABLE IF NOT EXISTS Customers (
    Customer_ID INTEGER PRIMARY KEY,
    Customer_Name TEXT NOT NULL,
    City TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS Products (
    Product_ID INTEGER PRIMARY KEY,
    Product_Name TEXT NOT NULL,
    Category TEXT NOT NULL,
    Unit_Price NUMERIC NOT NULL CHECK (Unit_Price >= 0),
    Stock_Quantity INTEGER NOT NULL DEFAULT 0
        CHECK (Stock_Quantity >= 0)
);

CREATE TABLE IF NOT EXISTS Orders (
    Order_ID INTEGER PRIMARY KEY,
    Customer_ID INTEGER NOT NULL REFERENCES Customers(Customer_ID),
    Order_Date TEXT NOT NULL,
    Total_Amount NUMERIC NOT NULL CHECK (Total_Amount >= 0)
);

CREATE TABLE IF NOT EXISTS Order_Items (
    Order_Item_ID INTEGER PRIMARY KEY,
    Order_ID INTEGER NOT NULL REFERENCES Orders(Order_ID),
    Product_ID INTEGER NOT NULL REFERENCES Products(Product_ID),
    Quantity INTEGER NOT NULL CHECK (Quantity > 0),
    Unit_Price NUMERIC NOT NULL CHECK (Unit_Price >= 0)
);
""")

pd.read_sql_query("""
SELECT name AS Table_Name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
""", conn)

,Table_Name
0,Customers
1,Order_Items
2,Orders
3,Products


## 3. Generate and verify synthetic data

In [3]:
rng = random.Random(42)

customers = [
    (
        i,
        f"Customer {i:03d}",
        rng.choice(["Johannesburg", "Cape Town", "Durban", "Pretoria"])
    )
    for i in range(1, 101)
]

products = [
    (1, "Keyboard", "Accessories", 350, 40),
    (2, "Mouse", "Accessories", 150, 60),
    (3, "Monitor", "Electronics", 2500, 20),
    (4, "Webcam", "Electronics", 600, 30),
    (5, "Laptop", "Electronics", 12000, 15),
    (6, "Headphones", "Accessories", 800, 35),
    (7, "Desk", "Furniture", 2200, 12),
    (8, "Office Chair", "Furniture", 1800, 18),
    (9, "USB Hub", "Accessories", 300, 50),
    (10, "Desk Lamp", "Furniture", 450, 25),
    (11, "Printer", "Electronics", 3000, 10),
    (12, "Standing Desk", "Furniture", 4500, 8)
]

orders = []
items = []
item_id = 1
start_date = date(2025, 1, 1)

for order_id in range(1, 501):
    customer_id = rng.randint(1, 90)
    order_date = start_date + timedelta(days=rng.randrange(730))
    product_ids = rng.sample(range(1, 11), rng.randint(1, 4))
    order_total = 0

    for product_id in product_ids:
        quantity = rng.randint(1, 5)
        unit_price = products[product_id - 1][3]

        items.append(
            (item_id, order_id, product_id, quantity, unit_price)
        )
        order_total += quantity * unit_price
        item_id += 1

    orders.append(
        (order_id, customer_id, order_date.isoformat(), order_total)
    )

existing_rows = sum(
    conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    for table in ["Customers", "Products", "Orders", "Order_Items"]
)

if existing_rows:
    print("Data already exists. No rows inserted.")
else:
    with conn:
        conn.executemany(
            "INSERT INTO Customers VALUES (?, ?, ?)", customers
        )
        conn.executemany(
            "INSERT INTO Products VALUES (?, ?, ?, ?, ?)", products
        )
        conn.executemany(
            "INSERT INTO Orders VALUES (?, ?, ?, ?)", orders
        )
        conn.executemany(
            "INSERT INTO Order_Items VALUES (?, ?, ?, ?, ?)", items
        )

    print("Practice data inserted successfully.")

display(pd.read_sql_query("""
SELECT 'Customers' AS Table_Name, COUNT(*) AS Row_Count FROM Customers
UNION ALL
SELECT 'Products', COUNT(*) FROM Products
UNION ALL
SELECT 'Orders', COUNT(*) FROM Orders
UNION ALL
SELECT 'Order_Items', COUNT(*) FROM Order_Items;
""", conn))

Data already exists. No rows inserted.


,Table_Name,Row_Count
0,Customers,100
1,Products,12
2,Orders,500
3,Order_Items,1237


## 4. Validate order totals

Compare order totals with summed order-item revenue. An empty result indicates no mismatches or orders without item totals.

In [4]:
query = """
WITH Order_Reconciliation AS (SELECT o.Order_ID,o.Total_Amount,SUM(oi.Quantity * oi.Unit_Price) AS Item_Total
FROM Orders o
LEFT JOIN Order_Items oi ON o.Order_ID = oi.Order_ID
GROUP BY o.Order_ID,o.Total_Amount)
SELECT * FROM Order_Reconciliation
WHERE Total_Amount <> Item_Total OR Item_Total IS NULL
"""

pd.read_sql_query(query, conn)

,Order_ID,Total_Amount,Item_Total


**Validation result:** No mismatched order totals or orders without item totals were found.

## 5. Monthly revenue and month-on-month change

Summarise 2026 revenue, order volume, and average order value. `LAG` compares each month with the preceding observed month; all twelve months are present in this dataset.

In [5]:
query = """
WITH Monthly_Summary AS (SELECT strftime('%Y-%m',Order_Date) AS Order_Month,COUNT(*) AS Order_Count,
COALESCE(SUM(Total_Amount),0) AS monthly_Revenue,ROUND(AVG(Total_Amount),2) AS Average_Order_Value
FROM Orders
WHERE Order_Date >= '2026-01-01' AND Order_Date < '2027-01-01'
GROUP BY Order_Month)
SELECT *,LAG(Monthly_Revenue) OVER (ORDER BY Order_Month) AS Previous_Month_Revenue,
monthly_Revenue - LAG(Monthly_Revenue) OVER (ORDER BY Order_Month) AS Revenue_Change,
ROUND(100.0 * (monthly_Revenue - LAG(Monthly_Revenue) OVER (ORDER BY Order_Month)) / NULLIF(LAG(Monthly_Revenue) OVER (ORDER BY Order_Month),0),2) AS
Revenue_Change_Percentage
FROM Monthly_Summary
ORDER BY Order_Month
"""

pd.read_sql_query(query, conn)

,Order_Month,Order_Count,monthly_Revenue,Average_Order_Value,Previous_Month_Revenue,Revenue_Change,Revenue_Change_Percentage
0,2026-01,22,353000,16045.45,NaN,NaN,NaN
1,2026-02,21,369200,17580.95,353000.0,16200.0,4.59
2,2026-03,20,216050,10802.50,369200.0,-153150.0,-41.48
3,2026-04,26,481600,18523.08,216050.0,265550.0,122.91
4,2026-05,10,185450,18545.00,481600.0,-296150.0,-61.49
5,2026-06,19,189700,9984.21,185450.0,4250.0,2.29
6,2026-07,17,274750,16161.76,189700.0,85050.0,44.83
7,2026-08,26,486700,18719.23,274750.0,211950.0,77.14
8,2026-09,26,570800,21953.85,486700.0,84100.0,17.28
9,2026-10,25,351000,14040.00,570800.0,-219800.0,-38.51


Revenue fell most in rand from April to May, decreasing by R296,150 (61.49%). The largest percentage decline occurred from November to December at 67.68%, because its R273,350 loss represented a larger share of November’s revenue.

## 6. Product performance

Include all products, with zeros for products without 2026 sales.

In [6]:
query = """
WITH Orders_2026 AS (SELECT oi.Product_ID,oi.Quantity,oi.Unit_Price,oi.Order_ID
FROM Orders o
JOIN Order_Items oi ON o.Order_ID = oi.Order_ID
WHERE o.Order_Date >= '2026-01-01' AND o.Order_Date < '2027-01-01')

SELECT p.Product_ID,p.Product_Name,p.Category,COALESCE(SUM(s.Quantity * s.Unit_Price),0) AS Revenue_2026,COALESCE(SUM(s.Quantity),0) AS units_2026,
COUNT(DISTINCT s.Order_ID) AS Orders_2026
FROM Products p 
LEFT JOIN Orders_2026 s ON s.Product_ID = p.Product_ID
GROUP BY p.Product_ID,p.Product_Name,p.Category
ORDER BY Revenue_2026 DESC
"""

pd.read_sql_query(query, conn)

,Product_ID,Product_Name,Category,Revenue_2026,units_2026,Orders_2026
0,5,Laptop,Electronics,2352000,196,65
1,3,Monitor,Electronics,497500,199,63
2,8,Office Chair,Furniture,369000,205,68
3,7,Desk,Furniture,330000,150,57
4,6,Headphones,Accessories,156800,196,65
5,4,Webcam,Electronics,94800,158,52
6,10,Desk Lamp,Furniture,77850,173,55
7,1,Keyboard,Accessories,56000,160,53
8,9,USB Hub,Accessories,55800,186,59
9,2,Mouse,Accessories,22950,153,50


Laptop sells for R12,000 per unit, compared with R1,800 for Office Chair. That higher price outweighs the slightly lower unit sales

## 7. Category performance and revenue shares

Count distinct orders within each category. An order can include multiple categories, so category order counts should not be added to obtain the overall order count.

In [13]:
query = """
WITH Orders_2026 AS (SELECT oi.Product_ID,oi.Quantity,oi.Unit_Price,oi.Order_ID
FROM Orders o
JOIN Order_Items oi ON o.Order_ID = oi.Order_ID
WHERE o.Order_Date >= '2026-01-01' AND o.Order_Date < '2027-01-01'),

Product_Summary AS (SELECT p.Category,COALESCE(SUM(s.Quantity * s.Unit_Price),0) AS Revenue_2026,COALESCE(SUM(s.Quantity),0) AS units_2026,
COUNT(DISTINCT s.Order_ID) AS Orders_2026
FROM Products p 
LEFT JOIN Orders_2026 s ON s.Product_ID = p.Product_ID
GROUP BY p.Category)
SELECT *,ROUND(100.0 * Revenue_2026 / NULLIF(SUM(Revenue_2026) OVER (),0),2) AS Revenue_Percentage,
ROUND(1.0 * Revenue_2026 / NULLIF(units_2026, 0), 2)
    AS Average_Selling_Price 
FROM Product_Summary
ORDER BY Revenue_2026 DESC
"""

pd.read_sql_query(query, conn)

,Category,Revenue_2026,units_2026,Orders_2026,Revenue_Percentage,Average_Selling_Price
0,Electronics,2944300,553,141,73.37,5324.23
1,Furniture,776850,528,135,19.36,1471.31
2,Accessories,291550,695,159,7.27,419.50


Electronics generated 73.37% of revenue from 553 units, while Accessories generated 7.27% from 695 units. Electronics’ higher average selling price per unit explains why it earned more revenue despite selling fewer units.

## 8. Customer participation

Preserve customers without 2026 orders when calculating participation.

In [8]:
query = """
WITH Customer_Summary AS (SELECT c.Customer_ID,c.Customer_Name,
c.City,COUNT(o.Order_ID) AS Orders_2026,
COALESCE(SUM(o.Total_Amount),0) AS Spending_2026
FROM Customers c
LEFT JOIN Orders o ON o.Customer_ID = c.Customer_ID
AND o.Order_Date >= '2026-01-01' AND o.Order_Date < '2027-01-01' 
GROUP BY c.Customer_ID,c.Customer_Name,c.City)
SELECT COUNT(*) AS Total_Customers,COALESCE(SUM(CASE WHEN Orders_2026 > 0 THEN 1 ELSE 0 END),0) AS Customers_With_Orders,
COALESCE(SUM(CASE WHEN Orders_2026 = 0 THEN 1 ELSE 0 END),0) AS Customers_Without_Orders
FROM Customer_Summary
"""

pd.read_sql_query(query, conn)

,Total_Customers,Customers_With_Orders,Customers_Without_Orders
0,100,81,19


## 9. City revenue and order behaviour

In [10]:
query = """
WITH Customer_Summary AS (SELECT c.Customer_ID,c.Customer_Name,
c.City,COUNT(o.Order_ID) AS Orders_2026,
COALESCE(SUM(o.Total_Amount),0) AS Spending_2026
FROM Customers c
LEFT JOIN Orders o ON o.Customer_ID = c.Customer_ID
AND o.Order_Date >= '2026-01-01' AND o.Order_Date < '2027-01-01' 
GROUP BY c.Customer_ID,c.Customer_Name,c.City)
SELECT City,COUNT(*) AS Total_Customers,COALESCE(SUM(CASE WHEN Orders_2026 > 0 THEN 1 ELSE 0 END),0) AS Customers_With_Orders,
COALESCE(SUM(Spending_2026),0)AS Revenue_2026,
ROUND(100.0 * SUM(CASE WHEN Orders_2026 > 0 THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0),2) AS Customer_Order_Rate,
COALESCE(SUM(Orders_2026),0) AS Total_Orders_2026,
ROUND(
    1.0 * SUM(Spending_2026)
        / NULLIF(SUM(Orders_2026), 0),
    2
) AS Average_Order_Value
FROM Customer_Summary
GROUP BY City
ORDER BY Revenue_2026 DESC
"""

pd.read_sql_query(query, conn)

,City,Total_Customers,Customers_With_Orders,Revenue_2026,Customer_Order_Rate,Total_Orders_2026,Average_Order_Value
0,Cape Town,29,24,1474850,82.76,76,19405.92
1,Johannesburg,27,21,972700,77.78,52,18705.77
2,Durban,24,20,919950,83.33,64,14374.22
3,Pretoria,20,16,645200,80.00,46,14026.09


In the synthetic 2026 data, Cape Town generated more revenue than Durban, despite Durban’s higher customer order rate. Cape Town recorded more orders (76 versus 64) and a higher average order value (R19,405.92 versus R14,374.22).

## 10. Top customers within each city

Rank by city and calculate revenue shares before selecting the top two distinct spending levels. Ties can return more than two customers per city.

In [11]:
query = """
WITH Customer_Summary AS (SELECT c.Customer_ID,c.Customer_Name,
c.City,COUNT(o.Order_ID) AS Orders_2026,
COALESCE(SUM(o.Total_Amount),0) AS Spending_2026
FROM Customers c
LEFT JOIN Orders o ON o.Customer_ID = c.Customer_ID
AND o.Order_Date >= '2026-01-01' AND o.Order_Date < '2027-01-01' 
GROUP BY c.Customer_ID,c.Customer_Name,c.City),
Customer_Ranks AS (SELECT Customer_ID,Customer_Name,City,Spending_2026,DENSE_RANK() OVER (PARTITION BY City ORDER BY Spending_2026 DESC) AS Customer_Rank,
ROUND(100.0 * Spending_2026 / NULLIF(SUM(Spending_2026) OVER (PARTITION BY City),0),2) AS City_Spending_Percentage
FROM Customer_Summary
)
SELECT * FROM Customer_Ranks
WHERE Customer_Rank <= 2
ORDER BY City, Customer_Rank, Customer_ID;
"""

pd.read_sql_query(query, conn)

,Customer_ID,Customer_Name,City,Spending_2026,Customer_Rank,City_Spending_Percentage
0,82,Customer 082,Cape Town,159150,1,10.79
1,18,Customer 018,Cape Town,156600,2,10.62
2,47,Customer 047,Durban,115100,1,12.51
3,66,Customer 066,Durban,104650,2,11.38
4,12,Customer 012,Johannesburg,177800,1,18.28
5,32,Customer 032,Johannesburg,128150,2,13.17
6,9,Customer 009,Pretoria,124200,1,19.25
7,53,Customer 053,Pretoria,117850,2,18.27


**Customer concentration:** Pretoria’s top two customers contributed 37.52% of its 2026 revenue, indicating dependence on a small number of buyers in this simulated dataset.

## 11. Stocked products without 2026 sales

In [12]:
query = """
SELECT
    p.Product_ID,
    p.Product_Name,
    p.Category,
    p.Stock_Quantity,
    p.Stock_Quantity * p.Unit_Price AS Unsold_Stock_Value
FROM Products AS p
WHERE p.Stock_Quantity > 0
  AND NOT EXISTS (
      SELECT 1
      FROM Order_Items AS oi
      JOIN Orders AS o
          ON o.Order_ID = oi.Order_ID
      WHERE oi.Product_ID = p.Product_ID
        AND o.Order_Date >= '2026-01-01'
        AND o.Order_Date < '2027-01-01'
  )
ORDER BY Unsold_Stock_Value DESC;
"""

pd.read_sql_query(query, conn)

,Product_ID,Product_Name,Category,Stock_Quantity,Unsold_Stock_Value
0,12,Standing Desk,Furniture,8,36000
1,11,Printer,Electronics,10,30000


Standing Desk and Printer had no sales in 2026 despite having stock available, representing R66,000 at current selling prices; however, this was deliberately built into the synthetic dataset and does not demonstrate weak demand.

## 12. Key findings

1. Monthly performance: September 2026 generated the highest monthly revenue at R570,800.
2. Product and category performance: Laptop was the highest-revenue product at R2,352,000. Electronics led all categories with R2,944,300, representing 73.37% of total 2026 revenue.
3. City performance: Cape Town generated more revenue than Durban despite Durban’s higher customer order rate. Cape Town recorded more orders (76 versus 64) and a higher average order value (R19,405.92 versus R14,374.22).
4. Unsold stock: Standing Desk and Printer had no 2026 sales despite available stock, representing R66,000 at current selling prices. Their exclusion from sales was deliberately built into the dataset and does not demonstrate weak demand.

## 13. Limitations

This project uses reproducible synthetic data to demonstrate SQL querying, validation, and interpretation. Findings describe the simulated dataset and should not be treated as evidence of real market behaviour. Stock quantities are a separate snapshot and were not reduced by simulated sales.

Customers 91–100 and products 11–12 were deliberately excluded from generated orders. Selling prices are fixed per product; the data does not model discounts, returns, costs, profit, or stock movements. These design choices limit the business conclusions.
